# CMPE 258 Team 7: hardened benchmark runs (Colab T4)

**Robust Neural Image Watermarking with Failure-Guided Attack Sampling (FGAS)**

Official numbers for the hardened benchmark come from this notebook on a **Colab T4 GPU**. Two teammates run it on their own Colab accounts at the same time, each with a different `MY_JOBS` list (section 3). Every run writes straight to the team Drive folder, so a Colab disconnect loses at most one epoch: re-running the job cell resumes from `last.pt`.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, set `MY_JOBS`, then `Runtime -> Run all`.

What it does:
1. Clones the repo, installs dependencies, runs the unit tests.
2. Data: COCO val2017 (4500 train / 500 val) for training and model selection; DIV2K valid HR (100 images) as the unseen test set. DIV2K train is only needed for the continuation experiment (D). Zips are cached on Drive.
3. Runs the selected jobs: calibration (`calib_*`), continuation from the DIV2K baseline (`*_continue_s*`), and the main from-scratch comparison (`hard_uniform_s*` vs `hard_fgas_s*`).
4. Severity sweep on the DIV2K test set, multi-seed aggregation, and the success-criterion check.
5. Collects the small result files into `results/checkin3/` and copies them to Drive for committing.

In [ ]:
# Fail fast on a CPU runtime: Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all.
import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU attached. Switch to Runtime -> Change runtime type -> T4 GPU and run again.')
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Team Drive folder: dataset zip cache, run folders (checkpoints, logs) and collected results.
# Change DRIVE_DIR if the shared team folder lives somewhere else in your Drive.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/CMPE258_Team7'
RUNS_DIR = f'{DRIVE_DIR}/runs_hardened'
import os
os.makedirs(f'{DRIVE_DIR}/cache', exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
REPO_URL = 'https://github.com/TaehunLeee5/CMPE258_Team7.git'
BRANCH = 'main'
%cd /content
!test -d CMPE258_Team7 || git clone -b {BRANCH} {REPO_URL}
%cd /content/CMPE258_Team7
!git pull --ff-only && git log -1 --oneline
!pip install -q -r requirements.txt
!python -m pytest -q

## 1. Data
COCO val2017 is split by sorted filename (first 4500 train, last 500 val) and downscaled to a short side of 256, stored as PNG. DIV2K valid HR is the held-out test set: it is never used for training, checkpoint selection or FGAS.

In [ ]:
!python scripts/download_coco.py --config configs/hard_base.yaml --cache-dir {DRIVE_DIR}/cache
!python scripts/prepare_coco.py --config configs/hard_base.yaml
!python scripts/validate_coco.py --config configs/hard_base.yaml

In [ ]:
# DIV2K: valid HR is the test set for every hardened run. Train HR is only needed for the
# continuation experiment (D), which starts from the committed DIV2K baseline checkpoint.
NEED_DIV2K_TRAIN = True
splits = 'train valid' if NEED_DIV2K_TRAIN else 'valid'
!python scripts/download_div2k.py --config configs/baseline.yaml --splits {splits} --cache-dir {DRIVE_DIR}/cache
if NEED_DIV2K_TRAIN:
    !python scripts/prepare_div2k.py --config configs/baseline.yaml
else:
    !python scripts/prepare_div2k.py --config configs/baseline.yaml --splits val

## 2. Jobs
| Job | Config | What |
|---|---|---|
| `calib_w100`, `calib_w200`, `calib_w400` | `configs/calib_w*.yaml` | 20-epoch uniform runs with final image-loss weight 100 / 200 / 400 |
| `calib_w100_strong` | `configs/calib_w100_strong.yaml` | stronger noise / JPEG / crop, only if every attack stays >= 99% |
| `baseline_continue_s{42,43,44}`, `fgas_continue_s{42,43,44}` | `configs/baseline_continue.yaml`, `configs/fgas.yaml` | Experiment D: 10 more epochs from the DIV2K baseline, uniform vs FGAS |
| `hard_uniform_s{42,43,44}`, `hard_fgas_s{42,43,44}` | `configs/hard_uniform.yaml`, `configs/hard_fgas.yaml` | Experiment E: 60 epochs from scratch, uniform vs FGAS (image-loss weight set from calibration) |
| `hard_fgas_p2_s{42,43,44}` | `configs/hard_fgas_p2.yaml` | Pre-registered fallback (FGAS with power 2), only if E misses the success criterion |

Split the jobs between the two Colab accounts so nobody runs the same job twice. A job that already finished is skipped, and an interrupted one resumes from its `last.pt` on Drive.

In [ ]:
BASELINE_CKPT = 'results/baseline_uniform/best.pt'
ALL_JOBS = {
    'calib_w400': ('configs/calib_w400.yaml', 42, []),
    'calib_w100': ('configs/calib_w100.yaml', 42, []),
    'calib_w200': ('configs/calib_w200.yaml', 42, []),
    'calib_w100_strong': ('configs/calib_w100_strong.yaml', 42, []),
}
for seed in (42, 43, 44):
    ALL_JOBS[f'baseline_continue_s{seed}'] = ('configs/baseline_continue.yaml', seed, [f'train.init_checkpoint={BASELINE_CKPT}'])
    ALL_JOBS[f'fgas_continue_s{seed}'] = ('configs/fgas.yaml', seed, [f'train.init_checkpoint={BASELINE_CKPT}'])
    ALL_JOBS[f'hard_uniform_s{seed}'] = ('configs/hard_uniform.yaml', seed, [])
    ALL_JOBS[f'hard_fgas_s{seed}'] = ('configs/hard_fgas.yaml', seed, [])
    ALL_JOBS[f'hard_fgas_p2_s{seed}'] = ('configs/hard_fgas_p2.yaml', seed, [])  # fallback, only if needed

# Edit per account, e.g. ['calib_w100', 'calib_w400'] on one and ['calib_w200', 'baseline_continue_s42', ...] on the other.
MY_JOBS = []

In [ ]:
import csv, pathlib, sys
sys.path.insert(0, 'src')
from wm.config import load_config


def finished(name, config, overrides):
    metrics = pathlib.Path(RUNS_DIR) / name / 'metrics.csv'
    if not metrics.exists():
        return False
    epochs = load_config(config, overrides)['train']['epochs']
    rows = list(csv.DictReader(open(metrics)))
    return bool(rows) and int(rows[-1]['epoch']) >= epochs


def run_job(name):
    config, seed, extra = ALL_JOBS[name]
    overrides = [f'experiment.name={name}', f'experiment.seed={seed}', f'experiment.output_dir={RUNS_DIR}', *extra]
    if finished(name, config, overrides):
        print(f'{name}: already finished, skipping')
        return
    last = pathlib.Path(RUNS_DIR) / name / 'last.pt'
    if last.exists():
        overrides.append(f'train.resume={last}')
        print(f'{name}: resuming from {last}')
    sets = ' '.join(overrides)
    get_ipython().system(f'python scripts/train.py --config {config} --set {sets}')
    if not finished(name, config, overrides):
        raise RuntimeError(f'{name} did not finish; re-run this cell to resume it')


for job in MY_JOBS:
    run_job(job)

## 3. Severity sweep on the DIV2K test set
Every attack at 4-5 levels, from mild to beyond the training strength (`attacks.sweep` in `configs/hard_base.yaml`), for every finished main-comparison run.

In [ ]:
import glob
for ckpt in sorted(glob.glob(f'{RUNS_DIR}/hard_*_s*/best.pt')):
    run_dir = os.path.dirname(ckpt)
    if not os.path.exists(f'{run_dir}/sweep.csv'):
        !python scripts/sweep.py --checkpoint {ckpt} --out {run_dir}/sweep.csv
sweeps = ' '.join(sorted(glob.glob(f'{RUNS_DIR}/hard_*_s*/sweep.csv')))
if sweeps:
    !python scripts/sweep.py --plot {sweeps} --out results/checkin3/figures/severity_sweep.png

## 4. Tables: calibration, continuation (D), main comparison (E)
Mean +- std over seeds. The criterion check compares FGAS with the matched uniform baseline on the DIV2K test set: worst-case attack bit accuracy +2 pp, PSNR within 0.5 dB, mean attacked accuracy not lower by more than 0.5 pp, clean accuracy >= 98%.

In [ ]:
OUT = 'results/checkin3'
!python scripts/aggregate_seeds.py {RUNS_DIR}/calib_* --file best_eval.json --no-group --out {OUT}/calibration/summary
!python scripts/aggregate_seeds.py {RUNS_DIR}/baseline_continue_s* {RUNS_DIR}/fgas_continue_s* --file best_eval.json --out {OUT}/exp_d
!python scripts/aggregate_seeds.py {RUNS_DIR}/hard_uniform_s* {RUNS_DIR}/hard_fgas_s* --out {OUT}/main_comparison --criterion hard_uniform hard_fgas
if glob.glob(f'{RUNS_DIR}/hard_fgas_p2_s*'):
    !python scripts/aggregate_seeds.py {RUNS_DIR}/hard_uniform_s* {RUNS_DIR}/hard_fgas_p2_s* --out {OUT}/fallback_p2 --criterion hard_uniform hard_fgas_p2

## 5. Collect result files and copy them to Drive
Small files only: metrics, configs, environment info, evaluation JSONs, FGAS history, sweeps, and one `best.pt` per method (seed 42). Whoever commits downloads `results/checkin3/` from Drive into the repo.

In [ ]:
import shutil
KEEP = ['metrics.csv', 'config.yaml', 'run_info.json', 'best_eval.json', 'best_test_eval.json', 'fgas_history.json', 'sweep.csv']
for run_dir in sorted(glob.glob(f'{RUNS_DIR}/*')):
    name = os.path.basename(run_dir)
    dest = pathlib.Path(OUT) / 'runs' / name
    dest.mkdir(parents=True, exist_ok=True)
    for f in KEEP:
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dest / f)
    if name.endswith('_s42') and os.path.exists(f'{run_dir}/best.pt'):
        shutil.copy(f'{run_dir}/best.pt', dest / 'best.pt')
shutil.copytree(OUT, f'{DRIVE_DIR}/results_checkin3', dirs_exist_ok=True)
print('Copied', OUT, 'to', f'{DRIVE_DIR}/results_checkin3')
!find {OUT} -maxdepth 2 | sort | head -50